In [1]:
import pandas as pd
import numpy as np

EQUITY_FILE  = "Company_Equity__Data.xlsx"
SAMPLE_START = "2010-01-01"
SAMPLE_END   = "2024-12-31"

SHEET_SECTOR = {
    "Financials Equity Data":    "XLF",
    "Industrials Equity Data":   "XLI",
    "Healthcare Equity Data":    "XLV",
    "Consumer Disc Equity Data": "XLY",
    "Technology Equity Data":    "XLK",
    "Energy Equity Data":        "XLE",}

In [2]:
xl = pd.ExcelFile(EQUITY_FILE)
frames = []
for sheet, sector in SHEET_SECTOR.items():
    df = xl.parse(sheet)
    df = df.rename(columns={df.columns[0]: "Date"})
    df["Date"] = pd.to_datetime(df["Date"])
    long = df.melt(id_vars="Date", var_name="Firm", value_name="Price")
    long["Firm"] = long["Firm"].str.split(" US Equity").str[0].str.strip()
    long["Sector"] = sector
    frames.append(long)

px = (pd.concat(frames, ignore_index=True)
        .dropna(subset=["Price"])
        .sort_values(["Firm", "Date"])
        .reset_index(drop=True))

# EXE is in the equity file but is excluded from the final sample at the CDS-liquidity screen. 
px = px[px["Firm"] != "EXE"].reset_index(drop=True)

In [3]:
def build_rv(g):
    # Targets are forward means (average daily RV over the next h days, excluding the current day)
    g = g.sort_values("Date").copy()
    r = np.log(g["Price"]).diff()
    
    # Daily realised variance proxy, annualised (Corsi, 2009).
    g["RV_daily"]   = 252.0 * r.pow(2)
    
    # Backward-looking HAR components.
    g["RV_weekly"]  = g["RV_daily"].rolling(5,  min_periods=5).mean()
    g["RV_monthly"] = g["RV_daily"].rolling(22, min_periods=22).mean()
    
    # Rolling mean is realigned so the value sits on the forecast date.
    fwd = g["RV_daily"].shift(-1)
    g["Target_h1"]  = fwd.rolling(1,  min_periods=1 ).mean()
    g["Target_h5"]  = fwd.rolling(5,  min_periods=5 ).mean().shift(-(5 - 1))
    g["Target_h22"] = fwd.rolling(22, min_periods=22).mean().shift(-(22 - 1))
    return g

In [4]:
rv = (px.groupby("Firm", group_keys=False)
        .apply(build_rv, include_groups=False))

panel = px[["Date", "Firm", "Sector"]].reset_index(drop=True).join(
    rv[["RV_daily", "RV_weekly", "RV_monthly",
        "Target_h1", "Target_h5", "Target_h22"]].reset_index(drop=True))

In [5]:
cds   = pd.read_parquet("Firm_CDS.parquet")
macro = pd.read_parquet("Macro_Controls.parquet")

# CDS merge: Firm+Date
cds = cds.drop(columns=["Sector"])
panel = panel.merge(cds,   on=["Firm", "Date"], how="left")
panel = panel.merge(macro, on="Date",           how="left")

In [6]:
panel = panel[(panel["Date"] >= SAMPLE_START) & (panel["Date"] <= SAMPLE_END)]

panel = panel[[
    "Date", "Firm", "Sector",
    "RV_daily", "RV_weekly", "RV_monthly",
    "Target_h1", "Target_h5", "Target_h22",
    "CDS_level", "CDS_change_5d",
    "VIX", "Yield_slope", "BAA10Y",
]].sort_values(["Firm", "Date"]).reset_index(drop=True)

panel.to_parquet("Panel.parquet", index=False)